In [2]:
import sys
import torch
import cv2
import numpy as np
import pandas as pd
import os
from torchvision import transforms
from PIL import Image
import torch.nn as nn
from torchvision import models
from torchvision.models import EfficientNet_V2_L_Weights
from pynvml import nvmlInit, nvmlDeviceGetCount, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo, nvmlShutdown
from tqdm import tqdm
import pickle

In [4]:
import psutil
import platform

print("CPU:", platform.processor())
print("CPU cores:", psutil.cpu_count(logical=True))
print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))

CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
CPU cores: 16
RAM (GB): 31.67


In [5]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU name: NVIDIA T1200 Laptop GPU


In [6]:
def extract_frames_from_video(video_path, num_frames=60):
    """
    Extract a fixed number of frames evenly from a video.
    
    Args:
        video_path (str): Path to the video file.
        num_frames (int): Number of frames to extract.
        
    Returns:
        list: List of frames in RGB format (as numpy arrays).
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # Create indices evenly spaced through the video
    frame_idxs = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    
    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)  # Move to the desired frame
        ret, frame = cap.read()
        if not ret:
            break
        # Convert BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    
    cap.release()
    return frames

In [7]:
preprocess = transforms.Compose([
    transforms.Resize(600),          # Resize so that the smaller side is 600 pixels
    transforms.CenterCrop(600),      # Then center crop to 600x600
    transforms.ToTensor(),           # Convert image to tensor with shape (3, H, W) in [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [8]:
# Load EfficientNetV2-L pretrained on ImageNet21k
model = models.efficientnet_v2_l(weights=EfficientNet_V2_L_Weights.IMAGENET1K_V1)

# Remove the classification head by replacing it with an identity layer
model.classifier = nn.Identity()

# Set up device and multi-GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
        model = model.to(device)
    except Exception as exc:
        print(f"[Warning] CUDA initialization failed ({exc}). Falling back to CPU.")
        device = torch.device("cpu")
        model = model.to(device)
else:
    model = model.to(device)

# if device.type == "cuda" and torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)
# model = model.to(device)
# Set model to eval mode

model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [10]:
from pathlib import Path
import os
import pickle

FEATURE_BATCH_SIZE = 4
FEATURE_OUTPUT_DIR = Path("efficientNet_features")
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _collect_video_items(video_root, include_augmented=False):
    video_items = []
    for lbl in sorted(os.listdir(video_root), key=lambda value: int(value)):
        class_path = os.path.join(video_root, lbl)
        if not os.path.isdir(class_path):
            continue

        label = int(lbl)
        for video in sorted(os.listdir(class_path)):
            if video.endswith((".mp4", ".avi")):
                video_path = os.path.join(class_path, video)
                video_items.append((f"{lbl}/{video}", video_path, label))

        if include_augmented and label in (0, 1):
            aug_path = os.path.join(class_path, "Augmented")
            if os.path.isdir(aug_path):
                for video in sorted(os.listdir(aug_path)):
                    if video.endswith((".mp4", ".avi")):
                        video_path = os.path.join(aug_path, video)
                        video_items.append((f"{lbl}/Augmented/{video}", video_path, label))

    return video_items


def _checkpoint_path(output_path):
    output_file = Path(output_path)
    return str(output_file.with_name(f"{output_file.stem}_checkpoint.pkl"))


def _load_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            state = pickle.load(f)
        video_features = state.get("video_features", [])
        labels = state.get("labels", [])
        done_ids = set(state.get("done_ids", []))
        print(f"Resuming from {checkpoint_path}: {len(done_ids)} videos already completed")
        return video_features, labels, done_ids

    return [], [], set()


def _save_checkpoint(checkpoint_path, video_features, labels, done_ids):
    payload = {
        "video_features": video_features,
        "labels": labels,
        "done_ids": sorted(done_ids),
    }
    tmp_path = f"{checkpoint_path}.tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump(payload, f)
    os.replace(tmp_path, checkpoint_path)


def run_feature_extraction_split(split_name, video_root, output_features_path, output_labels_path, include_augmented=False, batch_size=FEATURE_BATCH_SIZE):
    checkpoint_path = _checkpoint_path(output_features_path)
    all_items = _collect_video_items(video_root, include_augmented=include_augmented)
    video_features, labels, done_ids = _load_checkpoint(checkpoint_path)
    pending_items = [item for item in all_items if item[0] not in done_ids]

    print(f"{split_name}: {len(all_items)} total videos, {len(done_ids)} already done, {len(pending_items)} remaining")

    for batch_start in range(0, len(pending_items), batch_size):
        batch_items = pending_items[batch_start:batch_start + batch_size]
        batch_number = batch_start // batch_size + 1
        for video_id, video_path, label in tqdm(batch_items, desc=f"{split_name} batch {batch_number}", leave=False):
            features = extract_features_from_video(video_path, model, preprocess, device)
            video_features.append(features)
            labels.append(label)
            done_ids.add(video_id)

        _save_checkpoint(checkpoint_path, video_features, labels, done_ids)
        print(f"{split_name}: checkpoint saved after {len(done_ids)} videos")

    with open(output_features_path, "wb") as f:
        pickle.dump(video_features, f)

    with open(output_labels_path, "wb") as f:
        pickle.dump(labels, f)

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"{split_name}: final outputs saved")

In [11]:
def extract_features_from_video(video_path, model, preprocess, device, num_frames=60):
    """
    Extract feature vectors for a video by processing a fixed number of frames
    through EfficientNetB7.
    
    Args:
        video_path (str): Path to the video file.
        model (torch.nn.Module): EfficientNetB7 model (with classifier replaced).
        preprocess (torchvision.transforms.Compose): Preprocessing transforms.
        device (torch.device): Device to run the model on.
        num_frames (int): Number of frames to sample.
        
    Returns:
        numpy.ndarray: Feature vectors for each frame of shape (num_frames, feature_dim)
    """
    # Extract frames from the video
    frames = extract_frames_from_video(video_path, num_frames=num_frames)
    if not frames or len(frames) == 0:
        print(f"[Warning] No frames extracted from {video_path}")
        return None
    
    # Convert frames (numpy arrays) to PIL images and then apply preprocessing
    pil_frames = [Image.fromarray(frame) for frame in frames]
    frame_tensors = [preprocess(img) for img in pil_frames]

    if len(frame_tensors) == 0:
        print(f"[Warning] No preprocessed frames from {video_path}")
        return None
    
    # Stack the frame tensors to create a batch of shape: (num_frames, 3, 600, 600)
    batch_tensor = torch.stack(frame_tensors).to(device)
    
    # Pass the entire batch through the model to extract features
    with torch.no_grad():
        features = model(batch_tensor)
    
    return features.cpu().numpy()

In [9]:
video_paths = "dataset/daisee-separated/Train/"

run_feature_extraction_split(
    split_name="train",
    video_root=video_paths,
    output_features_path="efficientNet_features/daisee_train_video_features_v2_aug_60_2.pkl",
    output_labels_path="efficientNet_features/daisee_train_labels_v2_aug_60_2.pkl",
    include_augmented=True,
    batch_size=FEATURE_BATCH_SIZE,
)

Resuming from efficientNet_features\daisee_train_video_features_v2_aug_60_2_checkpoint.pkl: 5668 videos already completed
train: 6634 total videos, 5668 already done, 966 remaining


train: checkpoint saved after 5672 videos


train: checkpoint saved after 5676 videos


train: checkpoint saved after 5680 videos


train: checkpoint saved after 5684 videos


...

train: checkpoint saved after 6620 videos


train: checkpoint saved after 6624 videos


train: checkpoint saved after 6628 videos


train: checkpoint saved after 6632 videos


train: checkpoint saved after 6634 videos
train: final outputs saved


In [9]:
video_paths = "dataset/daisee-separated/Validation/"

run_feature_extraction_split(
    split_name="validation",
    video_root=video_paths,
    output_features_path="efficientNet_features/daisee_val_video_features_v2_aug_60_2.pkl",
    output_labels_path="efficientNet_features/daisee_val_labels_v2_aug_60_2.pkl",
    include_augmented=False,
    batch_size=FEATURE_BATCH_SIZE,
)

Resuming from efficientNet_features\daisee_val_video_features_v2_aug_60_2_checkpoint.pkl: 1232 videos already completed
validation: 1720 total videos, 1232 already done, 488 remaining


validation: checkpoint saved after 1236 videos


validation: checkpoint saved after 1240 videos


validation: checkpoint saved after 1244 videos


validation: checkpoint saved after 1248 videos


...

validation: checkpoint saved after 1704 videos


validation: checkpoint saved after 1708 videos


validation: checkpoint saved after 1712 videos


validation: checkpoint saved after 1716 videos


validation: checkpoint saved after 1720 videos
validation: final outputs saved


In [12]:
video_paths = "dataset/daisee-separated/Test/"

run_feature_extraction_split(
    split_name="test",
    video_root=video_paths,
    output_features_path="efficientNet_features/daisee_test_video_features_v2_aug_60_2.pkl",
    output_labels_path="efficientNet_features/daisee_test_labels_v2_aug_60_2.pkl",
    include_augmented=False,
    batch_size=FEATURE_BATCH_SIZE,
)

Resuming from efficientNet_features\daisee_test_video_features_v2_aug_60_2_checkpoint.pkl: 1628 videos already completed
test: 1723 total videos, 1628 already done, 95 remaining


test: checkpoint saved after 1632 videos


test: checkpoint saved after 1636 videos


test: checkpoint saved after 1640 videos


test: checkpoint saved after 1644 videos


...

test: checkpoint saved after 1708 videos


test: checkpoint saved after 1712 videos


test: checkpoint saved after 1716 videos


test: checkpoint saved after 1720 videos


test: checkpoint saved after 1723 videos
test: final outputs saved
